In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("F1_Homework2").getOrCreate()

BASE_PATH = "/Volumes/gr5069/raw/f1_data"

def read_csv(filename):
    """Read a CSV from the shared F1 Volume with standard options."""
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{BASE_PATH}/{filename}")
    )

drivers      = read_csv("drivers.csv")
pit_stops    = read_csv("pit_stops.csv")
results      = read_csv("results.csv")
races        = read_csv("races.csv")
constructors = read_csv("constructors.csv")

Q1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

Logic: The pit_stops table has one row per stop with a milliseconds duration. Group by raceId + driverId and compute three aggregates in one pass. Join races and drivers to replace integer IDs with readable names.

In [0]:
q1 = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.round(F.avg("milliseconds"), 0).alias("avg_pit_ms"),
        F.max("milliseconds").alias("slowest_pit_ms"),
        F.min("milliseconds").alias("fastest_pit_ms"),
    )
    .join(races.select("raceId", "name", "year"), on="raceId", how="left")
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId", how="left"
    )
    .select(
        "year",
        F.col("name").alias("race"),
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        "avg_pit_ms",
        "slowest_pit_ms",
        "fastest_pit_ms",
    )
    .orderBy("year", "race", "driver")
)

display(q1)

How it works: .groupBy("raceId", "driverId") partitions data so each group is one driver's stops in one race. .agg() computes all three statistics in a single distributed pass. The two .join() calls swap IDs for human-readable text. F.concat_ws(" ", ...) builds a full name from forename and surname.

Q2. Rank order by finishing position the average time spent at the pit stop in each race.

Logic: Join per-driver pit averages with results (which has positionOrder). Within each race, assign a rank by pit speed using a Window — rank 1 = fastest average pit. This lets us compare pit performance rank vs finishing position rank.

In [0]:
avg_pit = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.round(F.avg("milliseconds"), 0).alias("avg_pit_ms"))
)

window_by_race = Window.partitionBy("raceId").orderBy("avg_pit_ms")

q2 = (
    results
    .select("raceId", "driverId", "positionOrder")
    .join(avg_pit, on=["raceId", "driverId"], how="inner")
    .join(races.select("raceId", "name", "year"), on="raceId", how="left")
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId", how="left"
    )
    .withColumn("pit_speed_rank", F.rank().over(window_by_race))
    .select(
        "year",
        F.col("name").alias("race"),
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        F.col("positionOrder").alias("finish_position"),
        "avg_pit_ms",
        "pit_speed_rank",
    )
    .orderBy("year", "race", "finish_position")
)

display(q2)

How it works: Window.partitionBy("raceId").orderBy("avg_pit_ms") scopes ranking to a single race, sorted fastest-first. F.rank() handles ties by giving them the same rank (unlike row_number which breaks ties arbitrarily). The inner join on ["raceId","driverId"] naturally excludes drivers with no pit data (e.g. some DNFs).

Q3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

Logic: The code column holds 3-letter abbreviations. Missing values arrive as null (already handled by nullValue="\\N" at read time) or as blank strings. I derive a code from the surname's first 3 characters, uppercased — exactly the FIA convention. If the surname is shorter than 3 characters we prepend the forename.

In [0]:
derived_code = F.upper(
    F.when(
        F.length("surname") >= 3,
        F.substring("surname", 1, 3)
    ).otherwise(
        F.substring(F.concat("surname", "forename"), 1, 3)
    )
)

is_missing = F.col("code").isNull() | (F.trim(F.col("code")) == "")

q3 = (
    drivers
    .withColumn(
        "code_filled",
        F.when(is_missing, derived_code).otherwise(F.col("code"))
    )
    .select("driverId", "forename", "surname", "code", "code_filled")
    .orderBy("driverId")
)

display(q3)

How it works: is_missing catches both null and blank strings. derived_code uses a nested F.when: if surname ≥ 3 chars take substring(surname, 1, 3); otherwise concatenate surname + forename and take the first 3. F.upper ensures uppercase output. The outer F.when applies the fill only where needed — existing codes are preserved by .otherwise(F.col("code")).

Q4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

Logic: Age = number of complete years between dob and race date, computed as floor(months_between / 12). A driver born 10-Jul-1985 racing on 09-Jul-2010 is 24, not 25 — their birthday hasn't occurred yet that day. This matches the standard sports-media definition. Within each race a Window finds the min/max age.

In [0]:
race_drivers = (
    results
    .select("raceId", "driverId")
    .join(
        races.select("raceId", "name", "year", "date"),
        on="raceId", how="left"
    )
    .join(
        drivers.select("driverId", "forename", "surname", "dob"),
        on="driverId", how="left"
    )
    .withColumn(
        "age",
        F.floor(
            F.months_between(F.col("date").cast("date"), F.col("dob").cast("date")) / 12
        ).cast("int")
    )
)

w_race = Window.partitionBy("raceId")

q4 = (
    race_drivers
    .withColumn("min_age", F.min("age").over(w_race))
    .withColumn("max_age", F.max("age").over(w_race))
    .withColumn(
        "category",
        F.when(F.col("age") == F.col("min_age"), F.lit("Youngest"))
         .when(F.col("age") == F.col("max_age"), F.lit("Oldest"))
    )
    .filter(F.col("category").isNotNull())
    .select(
        "year",
        F.col("name").alias("race"),
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        "dob",
        "date",
        "age",
        "category",
    )
    .orderBy("year", "race", "category")
)

display(q4)

How it works: .cast("date") ensures both columns are proper date types before F.months_between. F.floor(... / 12) converts fractional months to whole completed years. The Window w_race partitions by race so min/max age is computed only among that race's starters. The F.when tags extremes; .filter(isNotNull()) drops all non-extreme rows.

Q5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

Logic: For each race row we want a running tally of that driver's podium finishes up to and including that race. Create 0/1 indicator columns for each podium position, then apply a cumulative F.sum over a Window ordered by raceId.

In [0]:
results_flagged = (
    results
    .withColumn("is_win", (F.col("positionOrder") == 1).cast("int"))
    .withColumn("is_p2",  (F.col("positionOrder") == 2).cast("int"))
    .withColumn("is_p3",  (F.col("positionOrder") == 3).cast("int"))
)

w_driver_cumul = (
    Window
    .partitionBy("driverId")
    .orderBy("raceId")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q5 = (
    results_flagged
    .withColumn("cum_wins", F.sum("is_win").over(w_driver_cumul))
    .withColumn("cum_p2",   F.sum("is_p2").over(w_driver_cumul))
    .withColumn("cum_p3",   F.sum("is_p3").over(w_driver_cumul))
    .join(races.select("raceId", "name", "year"), on="raceId", how="left")
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId", how="left"
    )
    .select(
        "year",
        F.col("name").alias("race"),
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        F.col("positionOrder").alias("finish_position"),
        "cum_wins",
        "cum_p2",
        "cum_p3",
    )
    .orderBy("year", "race", "driver")
)

display(q5)

How it works: (positionOrder == 1).cast("int") produces a 1 for wins, 0 otherwise. .rowsBetween(unboundedPreceding, currentRow) makes the sum cumulative — it always spans from the driver's very first race through the current row. Partitioning by driverId keeps each driver's counter independent. Ordering by raceId (a monotonically increasing integer matching race chronology) advances the window in time.

Q6. Continue exploring the data by answering your own question.

Own question: Which constructor has the highest avg points per race, and how has dominance evolved?

Logic: A team fields two cars per race; we sum both drivers' points to get the team's haul per race, then average that across all races entered. A year-by-year breakdown for the top 5 reveals the shifting eras of dominance.

In [0]:
# Step 1 — team points per race (sum both drivers)
constructor_race_pts = (
    results
    .groupBy("raceId", "constructorId")
    .agg(F.sum("points").alias("race_points"))
    .join(races.select("raceId", "year"), on="raceId", how="left")
)

# Step 2 — all-time average across every race entered
all_time = (
    constructor_race_pts
    .groupBy("constructorId")
    .agg(
        F.round(F.avg("race_points"), 2).alias("avg_pts_per_race"),
        F.count("raceId").alias("races_entered"),
    )
    .join(
        constructors.select("constructorId", F.col("name").alias("constructor")),
        on="constructorId", how="left"
    )
    .orderBy(F.desc("avg_pts_per_race"))
)

display(all_time)

# Step 3 — year-by-year for the top 5 constructors
top5_ids = [r["constructorId"] for r in all_time.limit(5).collect()]

yearly = (
    constructor_race_pts
    .filter(F.col("constructorId").isin(top5_ids))
    .groupBy("constructorId", "year")
    .agg(F.round(F.avg("race_points"), 2).alias("avg_pts_per_race"))
    .join(
        constructors.select("constructorId", F.col("name").alias("constructor")),
        on="constructorId", how="left"
    )
    .orderBy("constructor", "year")
)

display(yearly)

How it works: The first groupBy collapses both drivers' results into a single team score per race. The second groupBy averages those per-race scores across all of history. .limit(5).collect() brings only 5 rows to the driver (very cheap) to extract the top-5 IDs for filtering. The year-by-year view exposes eras: Ferrari dominated the early 2000s, Red Bull in 2010–13 and again from 2022, Mercedes from 2014–21. Note: the points system changed dramatically in 2010 (winner 25 pts vs 10 previously), so raw averages aren't perfectly comparable across eras — a refinement would normalize by points available per season.